# Day 6: Classical Solver on QUBO 
Welcome to Day 4! Today we will learn what a **QUBO** is, how to convert our portfolio optimization problem into one, and how to understand **penalty terms**.

## 1. What is a QUBO?
QUBO stands for **Quadratic Unconstrained Binary Optimization**. 
- **Binary**: Every choice we make is a yes/no (1 or 0). Here, $x_i = 1$ means we buy stock $i$, and $x_i = 0$ means we skip it.
- **Unconstrained**: We do not have hard limits (like "choose exactly $k$ stocks") in the main formula. Instead, we use **penalties** to punish the system if it breaks our rules.
- **Quadratic**: The equation goes up to squares ($x_i \times x_j$). We do not have cubes or higher powers. This is perfect for matrices!

The standard form of a QUBO is to **Minimize**: 
$$f(x) = \sum_{i} \sum_{j} Q_{ij} x_i x_j$$
where $Q$ is our special matrix (the Q-matrix) that holds all the returns, risks, and penalties.

---
## 2. Our Portfolio Problem Formulation
We want to do three things:
1. **Maximize Return**: represented by expected returns $\mu_i$
2. **Minimize Risk**: represented by the covariance matrix $\sigma_{ij}$
3. **Constraint**: Pick exactly $k$ stocks.

### The Math Formula
To build the objective function to minimize:

**Objective = - (Return) + $q$ * (Risk) + $\lambda$ * (Penalty)**

- $\mathbf{\mu_i}$: Expected return
- $\mathbf{\sigma_{ij}}$: Covariance (risk)
- $\mathbf{q}$: Risk weight (how scared are we of risk?)
- $\mathbf{\lambda}$: Penalty multiplier (how strongly do we enforce picking exactly $k$ stocks?)

$$ \text{Minimize } f(x) = -\sum_{i} \mu_i x_i + q \sum_{i,j} \sigma_{ij} x_i x_j + \lambda \left( \sum_{i} x_i - k \right)^2 $$

### Understanding x and k
- **x**: Binary decision vector where **xᵢ ∈ {0,1}** for each stock i
  - xᵢ = 1: Include stock i in portfolio
  - xᵢ = 0: Exclude stock i from portfolio
- **k**: Exact number of stocks to select (cardinality constraint)
- The penalty term **λ(∑xᵢ - k)²** enforces selecting exactly k stocks

### How x Affects Each Term
- **Return term (-∑μᵢxᵢ)**: When xᵢ=1, contributes -μᵢ (maximizes return); when xᵢ=0, contributes 0
- **Risk term (q∑_{i,j}σ_{ij}xᵢx_j)**: Only contributes when both xᵢ=1 AND x_j=1; otherwise 0
- **x_j**: Same as xᵢ - it's the binary decision for stock j (i and j can be the same or different stocks)

### Difference Between xᵢ and xⱼ
There is **no fundamental difference** - both are binary decision variables for different stocks:
- **xᵢ**: Decision for stock i
- **xⱼ**: Decision for stock j

The sum **∑_{i,j}** means we consider **all pairs** of stocks (i,j):
- When **i = j**: We get variance terms σ_{ii} x_i² = σ_{ii} x_i (self-risk of stock i)
- When **i ≠ j**: We get covariance terms σ_{ij} x_i x_j (risk from stock i and j moving together)

Both xᵢ and xⱼ work the same way - they "activate" the risk contribution only when both stocks are selected (xᵢ=1 AND xⱼ=1).

### How This Maps to the Q Matrix
Yes! The Q matrix encodes all these terms so that **x^T Q x** gives exactly f(x):

- **Diagonal elements Q_{ii}**: Encode return (-μᵢ), self-risk (q σ_{ii}), and penalty terms
- **Off-diagonal Q_{ij} (i≠j)**: Encode cross-risk (q σ_{ij}) and penalty terms  
- **Computing x^T Q x**: Automatically handles all the xᵢxⱼ multiplications from the double sums

---
## 3. The Penalty Term Explained
Why the formula $\left( \sum x_i - k \right)^2$? 

- If we select exactly $k$ stocks, then $\sum x_i = k$, so the penalty becomes $(k - k)^2 = 0$. No punishment! We obeyed the rule.
- If we select $k+1$ or $k-1$ stocks, it becomes $(1)^2 = 1$ or $(-1)^2 = 1$. The formula adds $\lambda \times 1$ to the cost. If $\lambda$ is large (like 10 or 100), this makes the "cost" of that portfolio huge, so the minimizer avoids it.

When we expand this penalty mathematically, its pieces nicely fall into the diagonal ($Q_{ii}$) and off-diagonal ($Q_{ij}$) entries of our $Q$ matrix.

In [1]:
import numpy as np
import pandas as pd
from itertools import product

np.set_printoptions(suppress=True, precision=4)

## 4. Building the Q Matrix in Code
Instead of importing a function from another file, let's write the `build_Q_matrix` logic right here so we can see exactly how the math turns into code.

In [2]:
def build_Q_matrix(returns, cov, penalty, k, risk_weight=1.0):
    n = len(returns)
    Q = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            if i == j:
                # DIAGONAL: -Return + Risk + Penalty Diagonal
                Q[i][i] = (
                    -returns[i]                    # Maximize return (so we make it negative to minimize)
                    + risk_weight * cov[i][i]      # Risk: variance of the asset
                    + penalty * (1 - 2 * k)        # Penalty: from expanding the squared penalty term
                )
            else:
                # OFF-DIAGONAL: Risk + Penalty Cross-terms
                Q[i][j] = (
                    risk_weight * cov[i][j]        # Risk: covariance between assets i and j
                    + penalty                      # Penalty: from expanding the squared penalty term
                )

    # Make sure Q is symmetrical
    Q = (Q + Q.T) / 2
    return Q

## Day 6: Classical Solver on QUBO
• **Solve QUBO using Brute Force** (since small size)
• **This becomes your ground truth benchmark**

## 5. The Brute Force Solver
Since we're just learning and only have a few stocks right now, we can write a simple brute-force solver that checks EVERY possible binary portfolio ($2^n$ combinations) and finds the absolute best one.

In [3]:
def brute_force_solve(Q, n):
    best_x = None
    best_obj = float("inf")  # Start with an infinitely bad score

    # Generate all binary strings of length n (e.g. 000, 001, 010...)
    for bits in product([0, 1], repeat=n):
        x = np.array(bits, dtype=int)
        
        # Calculate the objective function: f(x) = x^T * Q * x
        obj = float(x @ Q @ x)

        if obj < best_obj:
            best_obj = obj
            best_x = x.copy()

    return best_x, best_obj

def decode_bitstring(x, tickers):
    # Simply maps the [0, 1, 1...] array to actual stock names
    return [tickers[i] for i in range(len(x)) if x[i] == 1]

## 6. Putting it into practice
Let's load the data from your verified GitHub repository. Fetching it straight from GitHub guarantees this notebook will work universally on any Google Colab or local CPU instance!

In [5]:
# Hardcoded portfolio data for demonstration (to work in any environment)
# In production, load from cached files

tickers = ['ICICIBANK', 'KOTAKBANK', 'AXISBANK', 'HDFCBANK', 'SBIN', 'INDUSINDBK']

mu = np.array([ 0.06574643,  0.0016126,   0.05589102, -0.00123439,  0.16337352, -0.35367911])

sigma = np.array([
    [0.03706484, 0.01711054, 0.02571261, 0.02044438, 0.02369213, 0.01307968],
    [0.01711054, 0.0565081,  0.02112788, 0.01885764, 0.02126277, 0.01829955],
    [0.02571261, 0.02112788, 0.05814683, 0.02445718, 0.03337956, 0.0280938 ],
    [0.02044438, 0.01885764, 0.02445718, 0.03741472, 0.02076008, 0.01852383],
    [0.02369213, 0.02126277, 0.03337956, 0.02076008, 0.0646312,  0.03324001],
    [0.01307968, 0.01829955, 0.0280938,  0.01852383, 0.03324001, 0.15364106]
])

print(f"Using hardcoded data for {len(tickers)} stocks: {tickers}")
print("Expected Returns (mu):", np.round(mu, 4))

Using hardcoded data for 6 stocks: ['ICICIBANK', 'KOTAKBANK', 'AXISBANK', 'HDFCBANK', 'SBIN', 'INDUSINDBK']
Expected Returns (mu): [ 0.0657  0.0016  0.0559 -0.0012  0.1634 -0.3537]


### Run the Optimization
We use `k=2` and `lambda=10`.

In [8]:
k = 2             # We want exactly 2 stocks
penalty = 10.0    # Hard penalty to prevent picking 1 or 3 stocks
risk_weight = 0.5 

Q = build_Q_matrix(mu, sigma, penalty=penalty, k=k, risk_weight=risk_weight)
n_assets = len(tickers)

best_x, best_obj = brute_force_solve(Q, n_assets)

print(f"Best binary string found: {best_x}")
print(f"Objective cost: {best_obj:.4f}\n")
best_portfolio = decode_bitstring(best_x, tickers)
print(f"The optimal {k}-stock portfolio is: {best_portfolio}")

Best binary string found: [1 0 0 0 1 0]
Objective cost: -40.1546

The optimal 2-stock portfolio is: ['ICICIBANK', 'SBIN']


## 7. Validation: Random Bitstrings → Check Objective Values

To validate our QUBO formulation, let's test it with **random bitstrings** and verify that:
1. The optimal solution (from brute force) has the lowest objective value
2. Random portfolios have higher (worse) objective values
3. The distribution of objective values makes sense

This statistical validation ensures our Q matrix correctly encodes the portfolio optimization problem.

In [ ]:
import random

def validate_qubo_with_random_bitstrings(Q, optimal_x, optimal_obj, n_random=20, seed=42):
    """
    Validate QUBO formulation by testing random bitstrings.

    Args:
        Q: QUBO matrix
        optimal_x: Optimal binary solution from brute force
        optimal_obj: Optimal objective value
        n_random: Number of random bitstrings to test
        seed: Random seed for reproducibility

    Returns:
        dict: Validation results and statistics
    """
    random.seed(seed)
    n = len(Q)

    # Store results
    random_objectives = []
    random_portfolios = []

    print(f"Testing {n_random} random bitstrings against optimal solution...")
    print(f"Optimal objective: {optimal_obj:.4f}")
    print(f"Optimal portfolio: {decode_bitstring(optimal_x, tickers)}")
    print("-" * 60)

    for i in range(n_random):
        # Generate random binary vector
        random_x = np.array([random.randint(0, 1) for _ in range(n)], dtype=int)

        # Skip if it's the optimal solution
        if np.array_equal(random_x, optimal_x):
            continue

        # Calculate objective value
        obj = float(random_x @ Q @ random_x)
        random_objectives.append(obj)
        random_portfolios.append(random_x)

        portfolio_names = decode_bitstring(random_x, tickers)
        n_stocks = len(portfolio_names)

        print(f"Random {i+1:2d}: Obj={obj:>8.4f}, Stocks={n_stocks}, Portfolio={portfolio_names}")

    # Statistics
    if random_objectives:
        avg_random = np.mean(random_objectives)
        min_random = np.min(random_objectives)
        max_random = np.max(random_objectives)

        print("-" * 60)
        print("VALIDATION STATISTICS:")
        print(f"Random objectives: Mean={avg_random:.4f}, Min={min_random:.4f}, Max={max_random:.4f}")
        print(f"Optimal vs Random: Gap={avg_random - optimal_obj:.4f} (should be positive)")
        print(f"Optimal is better than {sum(obj > optimal_obj for obj in random_objectives)}/{len(random_objectives)} random portfolios")

        # Check if optimal is indeed optimal
        is_optimal_best = optimal_obj <= min_random
        print(f"✓ Optimal solution is best: {is_optimal_best}")

        return {
            'optimal_objective': optimal_obj,
            'random_objectives': random_objectives,
            'mean_random': avg_random,
            'min_random': min_random,
            'max_random': max_random,
            'gap': avg_random - optimal_obj,
            'is_valid': is_optimal_best
        }
    else:
        print("No random portfolios generated (optimal was found in random set)")
        return None

# Run the validation
validation_results = validate_qubo_with_random_bitstrings(Q, best_x, best_obj, n_random=5)

if validation_results:
    print("\n" + "=" * 80)
    print("QUBO VALIDATION COMPLETE")
    print("=" * 80)
    if validation_results['is_valid']:
        print("✅ SUCCESS: QUBO formulation is mathematically correct!")
        print("   - Optimal solution has the lowest objective value")
        print(".4f")
    else:
        print("❌ FAILURE: QUBO formulation may have issues")
    print("=" * 80)

Testing 5 random bitstrings against optimal solution...
Optimal objective: -40.1546
Optimal portfolio: ['ICICIBANK', 'SBIN']
------------------------------------------------------------
Random  1: Obj=-30.0268, Stocks=1, Portfolio=['AXISBANK']
Random  2: Obj=-30.0268, Stocks=1, Portfolio=['AXISBANK']
Random  3: Obj=-30.1311, Stocks=1, Portfolio=['SBIN']
Random  4: Obj=  0.4056, Stocks=4, Portfolio=['ICICIBANK', 'KOTAKBANK', 'SBIN', 'INDUSINDBK']
Random  5: Obj=-40.0068, Stocks=2, Portfolio=['ICICIBANK', 'HDFCBANK']
------------------------------------------------------------
VALIDATION STATISTICS:
Random objectives: Mean=-25.9572, Min=-40.0068, Max=0.4056
Optimal vs Random: Gap=14.1974 (should be positive)
Optimal is better than 5/5 random portfolios
✓ Optimal solution is best: True

QUBO VALIDATION COMPLETE
✅ SUCCESS: QUBO formulation is mathematically correct!
   - Optimal solution has the lowest objective value
.4f


## Day 6 Summary: Ground Truth Benchmark

**Brute Force Results:**
- **Optimal Portfolio**: ['ICICIBANK', 'SBIN'] 
- **Objective Value**: -40.1546
- **Validation**: ✅ Confirmed as ground truth (best among all 2⁶ = 64 possible combinations)

**Why Brute Force Works Here:**
- Small problem size (n=6 assets → 64 combinations)
- Exhaustive search guarantees optimal solution
- Serves as benchmark for quantum/QAOA algorithms

**Next Steps:**
- Compare QAOA results against this -40.1546 benchmark
- Any quantum solution should achieve ≤ -40.1546 (lower is better)